# 생성기 v1: phrase graph 위 유형열 + 마스크형 이산 확산 + 그래프 손실

2026-11-11–12에 정한 설계를 구현한 실험입니다.

| 층 | 방법 |
|---|---|
| 구조 | 원곡의 phrase graph G 위에서 1차 Markov 보행으로 새 유형열 Y′를 뽑음. 시작·끝 유형 고정, 제1장(1–4마디)과 제4장(25–28마디)은 같은 유형 |
| 선율 | 마디를 격자(박당 4칸)로 적고, 유형을 조건으로 마스크형 이산 확산으로 생성. 제1장과 제4장은 한 변수로 묶어 공동 디코딩 |
| 손실 | 확산 ELBO(원곡 72마디) + λ · 그래프 손실 (i)(새 유형열을 조건으로 모두 가린 입력에서 계산). 그래프 손실은 step 0부터 켜고 50 step 동안 λ를 올림 |

네 학습 조건을 비교합니다.

| 조건 | λ |
|---|---|
| `diffusion` | 0 (확산 손실만) |
| `diffusion_graph_025` | 0.25 |
| `diffusion_graph_050` | 0.5 |
| `diffusion_graph` | 1 |

**λ 선택 규칙 (실행 전에 고정):** λ > 0인 조건 가운데 판정 기준 4(복제 검정 |z_novel| < 2)와 5(λ = 0보다 거부 전 유형 소속이 높음)를 학습 Seed 3개 모두에서 통과하는 것 중, 고른 학습 길이에서 보류 ELBO가 가장 낮은 것을 고릅니다. 판정 지표와 따로인 가능도로 고르기 위해서입니다.

학습 조건마다 학습 Seed 3개 × 보류 묶음 3개로 학습 길이를 고르고, 그 길이로 72마디 전체를 학습해 60곡을 생성합니다.
전체 학습 중에는 고른 학습 길이를 5등분한 시점마다 곡 4개를 끝까지 생성해, 표면의 phrase graph가 조건으로 준 유형열을 따르는지 기록합니다(감시용, 기울기 없음).

그 뒤 평가 단계가 이어집니다.
- **복제 검정용 생성**: 보류 모형(학습 Seed 3개 × 보류 묶음 3개)을 고른 길이까지 다시 학습해 각각 20곡을 생성합니다.
- **평가 계산**: 2026-11-12에 고정한 네 축(기하·통계·음악·새로움)과 판정 기준을 `evaluation.json`에 계산합니다.

**이번 실행은 우리 방법만 평가합니다.** 기준선(R0–R2, T0/T2)과 정재훈 교수팀 방법(TLJ-A)과의 비교는 아직 하지 않습니다. 따라서 판정 기준 중 5(그래프 손실 효과)와 4의 일부(복제 검정 |Z| < 2)만 λ > 0인 조건마다 판정하고, 나머지는 '비교 대기'로 둡니다.

**실행 순서**: Runtime → Change runtime type에서 **GPU**를 고른 뒤 위에서 아래로 실행합니다. 안내: `docs/chart_diffusion_colab.md`.

**덮어쓰기 없음**: 결과는 `artifacts/chart_diffusion_runs/<RUN_TAG>/` 아래에만 쓰입니다.

In [ ]:
# Cell 0 — 실험 전용 clone 준비 (Colab에서만 동작합니다)
# 기존 MyDrive/DiffDodree는 건드리지 않고, 실험 branch를 별도 폴더에 clone합니다.
from pathlib import Path
import importlib.util
import os
import subprocess

REPO_URL = "https://github.com/SHlee-TDA/DiffDodree.git"
BRANCH = "claude/method-section-model-improve-d5ad4d"
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/DiffDodree_chart")

try:
    _in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _in_colab = False

if _in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    def git(*args: str) -> str:
        return subprocess.check_output(["git", "-C", str(EXPERIMENT_ROOT), *args], text=True).strip()

    if not EXPERIMENT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(EXPERIMENT_ROOT)],
            check=True,
        )
    else:
        if git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError(f"{EXPERIMENT_ROOT}는 다른 저장소의 clone입니다.")
        if git("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(
                f"{EXPERIMENT_ROOT}에 commit하지 않은 수정이 있습니다. 덮어쓰지 않도록 여기서 멈춥니다."
            )
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(EXPERIMENT_ROOT), "merge", "--ff-only", f"origin/{BRANCH}"], check=True
        )
    print("Commit:", git("log", "-1", "--oneline"))
    os.environ["GUGAKGEN_PROJECT_ROOT"] = str(EXPERIMENT_ROOT)
    import sys

    if any(name == "gugakgen" or name.startswith("gugakgen.") for name in sys.modules):
        # 이전 실행에서 import한 코드가 메모리에 남아 있으면 방금 받은 commit이 아니라 그 코드로 돕니다.
        # (20261112_1955 실행은 meta에 f46935c가 적혔지만 실제로는 45618a9 코드로 돌았습니다.)
        raise RuntimeError("이 session에 이전 코드가 import되어 있습니다. Runtime → Restart session 후 다시 Run all 하세요.")
else:
    print("Colab이 아니므로 현재 project folder를 그대로 사용합니다.")

In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

In [ ]:
# 실험 code가 들어 있는 commit인지, GPU가 잡혔는지 확인합니다.
import subprocess

import torch

from gugakgen.workflows import chart_diffusion_experiment  # 없으면 branch가 잘못된 것입니다.

print("Commit:", subprocess.check_output(["git", "-C", str(project_root), "log", "-1", "--oneline"], text=True).strip())
print("Runtime:", runtime.as_dict())
if runtime.backend == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 연구자가 주로 바꾸는 값입니다. λ·Seed·step·격자 칸 수는 configs/chart_diffusion_v1.yaml에 있습니다.
from datetime import datetime, timezone, timedelta

RUN_MODE = "full"  # 코드 경로만 확인하려면 "smoke" (20 step, Seed 1개, 2곡)

# None이면 실행 시각으로 새 이름을 만듭니다.
# 연결이 끊겨 다시 실행할 때는 이전 실행에서 출력된 RUN_TAG를 그대로 넣으세요. 끝난 단위는 건너뜁니다.
RUN_TAG = None

CONFIG_PATH = project_root / "configs/chart_diffusion_v1.yaml"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'이어야 합니다.")
if RUN_MODE == "full" and runtime.backend != "cuda":
    raise RuntimeError("full 실행은 CUDA GPU에서만 합니다. Runtime → Change runtime type에서 GPU를 고르세요.")
if RUN_TAG is None:
    RUN_TAG = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M")
print("RUN_TAG =", repr(RUN_TAG), " ← 다시 실행할 때 이 값을 쓰세요.")

In [ ]:
# 실험 전체를 실행합니다. 학습 1회가 끝날 때마다 진행 상황을 출력합니다.
# 소요 시간은 아직 T4에서 재 보지 않았습니다. CPU에서는 그래프 손실을 켠 800 step 학습이 약 3분이었습니다.
# 마지막 평가 단계는 CPU 계산이며 모형 6개 기준으로 몇 분이 걸립니다(로컬 측정: 모형 1개 60곡에 약 30초 + 복제 검정).
from gugakgen.workflows.chart_diffusion_experiment import run_experiment

report = run_experiment(CONFIG_PATH, project_root, RUN_TAG, runtime, run_mode=RUN_MODE)
RUN_DIR = project_root / "artifacts/chart_diffusion_runs" / (RUN_TAG + ("" if RUN_MODE == "full" else "_smoke"))
print("저장:", RUN_DIR / "report.json")

In [ ]:
# 0) 유형화와 조건
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
print("원곡 유형열:", report["typology"]["sequence"])
print("환입(조건으로 묶은 마디):", report["returns"])
print("고른 학습 길이:", {arm: c["selected"] for arm, c in report["selected_steps"].items()})
print("그 길이에서 그래프 손실이 켜져 있던 step 수:",
      {arm: c.get("graph_active_steps", "기록 없음") for arm, c in report["selected_steps"].items()})

In [ ]:
# 1) 학습 길이에 따른 보류 ELBO (칸당 nat, 낮을수록 좋음)와 감시 지표
#    그림의 글자는 Colab에 한글 글꼴이 없어 영어로 적었습니다.
import matplotlib.pyplot as plt

from gugakgen.workflows.chart_diffusion_experiment import (
    curve_rows, generation_rows, heldout_baseline, monitor_rows, recall_rows,
)

curves = pd.DataFrame(curve_rows(report))
monitor = pd.DataFrame(monitor_rows(report))
baseline = heldout_baseline(report)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for arm, frame in curves.groupby("arm"):
    line, = axes[0].plot(frame["step"], frame["heldout_elbo"], marker="o", label=f"{arm} · held-out")
    axes[0].plot(frame["step"], frame["train_elbo"], linestyle=":", color=line.get_color(), label=f"{arm} · train")
    best = report["selected_steps"][arm]["selected"]
    axes[0].axvline(best, color=line.get_color(), alpha=0.4, linewidth=1)
if baseline == baseline:  # NaN for older runs
    axes[0].axhline(baseline, color="gray", linestyle="--", linewidth=1, label="context-free baseline")
for arm, frame in monitor.groupby("arm"):
    line, = axes[1].plot(frame["step"], frame["first_pass_membership"], marker="o", label=f"{arm} · type membership (before rejection)")
    axes[1].plot(frame["step"], frame["surface_edge_validity"], marker="s", linestyle="--", color=line.get_color(),
                 label=f"{arm} · surface edges in G")
    axes[2].plot(frame["step"], frame["original_phrase_rate"], marker="o", label=arm)
axes[0].set(xlabel="training step (log)", ylabel="ELBO per cell (nats)", xscale="log",
            title="held-out ELBO (vertical line = chosen length)")
axes[1].set(xlabel="training step", ylabel="share", ylim=(0, 1.02), title="structure realised by generated pieces (monitor)")
axes[2].set(xlabel="training step", ylabel="share", ylim=(0, 1.02), title="phrases copied from the original")
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
if monitor["step"].nunique() < 2:
    print("감시 시점이 하나뿐이라 가운데·오른쪽 그림은 점으로만 보입니다(학습 길이가 감시 간격보다 짧았던 실행).")


In [ ]:
# 2) 암기 정도: 모두 가린 입력에서 argmax로 원곡 마디를 그대로 내는 비율
display(pd.DataFrame(recall_rows(report)).set_index("arm"))

# 3) 생성 시 기록한 지표 (조건마다 학습 Seed 3개 × 60곡). 평가는 아래 5)–7)을 보세요.
display(pd.DataFrame(generation_rows(report)).set_index("arm").T)

In [ ]:
# 5) λ 선택과 판정 기준 (2026-11-12 고정). 비교 생성기가 필요한 기준은 '비교 대기'로 표시됩니다.
from gugakgen.workflows.chart_diffusion_experiment import (
    copytest_rows, criteria_rows, evaluation_rows, selection_rows, tradeoff_rows,
)

evaluation = report["evaluation"]
print("선택 규칙:", evaluation["selection"]["rule"])
display(pd.DataFrame(selection_rows(evaluation)).set_index("arm"))
print("고른 조건:", evaluation["selection"]["selected"] or "없음 (규칙을 만족하는 λ가 없음)",
      "· 아래 판정은", evaluation["criteria_arm"], "기준")
display(pd.DataFrame(criteria_rows(evaluation)).set_index("criterion"))
print("원곡 기준값:", evaluation["reference"])


In [ ]:
# 6) 네 축의 지표 (학습 Seed 3개의 평균과 범위). 곡 단위 95% 구간은 evaluation.json의 models[*].per_piece에 있습니다.
pd.set_option("display.max_rows", 60)
display(pd.DataFrame(evaluation_rows(evaluation)).set_index(["axis", "metric"]))

# 복제 검정 C_T: Z가 크게 음수이면 생성 마디가 보류 마디보다 학습 마디에 가깝다(복제). 0 근처가 바람직합니다.
# z: 생성 마디 전부. 원곡은 마디의 46%를 곡 안에서 반복하는데 보류 마디는 학습 마디와 같을 수 없어서 음수로 치우칩니다.
# z_novel: 학습 마디와 똑같은 마디를 뺀 나머지로 계산(판정 기준 4에 씀). 뺀 비율은 exact_train_share이고,
#          원곡 기준값의 repeat_rate(곡 안에서 반복되는 마디 비율)와 함께 읽습니다.
display(pd.DataFrame(copytest_rows(evaluation)))

In [ ]:
# 7) 새로움–충실도 평면: 가로 authenticity(복제가 아닌 마디 비율), 세로 density(원곡 분포 안쪽 정도).
#    오른쪽 위가 "새롭지만 원곡 양식 안"입니다. 점 하나가 학습 Seed 하나, 막대는 곡 단위 bootstrap 95% 구간입니다.
fig, ax = plt.subplots(figsize=(6, 5))
for arm in evaluation["arms"]:
    models = [m for m in evaluation["models"] if m["arm"] == arm]
    x = [m["set"]["authenticity"]["value"] for m in models]
    y = [m["set"]["density"]["value"] for m in models]
    xerr = [[v - m["set"]["authenticity"]["ci95"][0] for v, m in zip(x, models)],
            [m["set"]["authenticity"]["ci95"][1] - v for v, m in zip(x, models)]]
    yerr = [[v - m["set"]["density"]["ci95"][0] for v, m in zip(y, models)],
            [m["set"]["density"]["ci95"][1] - v for v, m in zip(y, models)]]
    ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="o", capsize=3, label=arm)
ax.set(xlabel="authenticity", ylabel="density", xlim=(0, 1.02), title="novelty vs fidelity (our arms only)")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()
# 8) λ에 따른 맞교환: 구조 실현(거부 전 유형 소속)과 새로움(복제 검정 z_novel). 점 하나가 학습 Seed 하나입니다.
trade = pd.DataFrame(tradeoff_rows(evaluation))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(trade["lambda"], trade["first_pass_membership"])
axes[0].plot(*zip(*trade.groupby("lambda")["first_pass_membership"].mean().items()), color="gray")
axes[0].set(xlabel="λ (graph loss weight)", ylabel="type membership before rejection", title="structure")
axes[1].scatter(trade["lambda"], trade["z_novel"])
axes[1].plot(*zip(*trade.groupby("lambda")["z_novel"].mean().items()), color="gray")
axes[1].axhline(-2, color="red", linestyle="--", linewidth=1, label="|Z| = 2")
axes[1].axhline(2, color="red", linestyle="--", linewidth=1)
axes[1].set(xlabel="λ (graph loss weight)", ylabel="copy test z_novel (Stouffer over folds)", title="novelty")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
display(trade.round(3))


In [ ]:
# 4) 들어 볼 악보: 조건마다 첫 학습 Seed의 첫 3곡을 MusicXML로 저장합니다(이미 있으면 멈춤).
from gugakgen.workflows.chart_diffusion_experiment import export_musicxml, load_setup

setup = load_setup(CONFIG_PATH, project_root)
listen_dir = RUN_DIR / "musicxml"
for unit in report["full"]:
    if unit["seed"] == report["full"][0]["seed"]:
        for path in export_musicxml(unit, setup, listen_dir):
            print(path.relative_to(project_root))

## 표 읽는 법

**1) 왼쪽: 보류 ELBO.** 학습에 쓰지 않은 마디를 얼마나 잘 설명하는지입니다. 내려가다 다시 오르는 지점부터 외우기 시작합니다.
전체 학습 길이는 이 곡선의 최저점(세로선)으로 정했습니다. 회색 점선은 문맥을 전혀 쓰지 않는 기준(같은 길이 마디의 같은 칸 분포)으로,
최저점이 이 선보다 아래여야 모형이 유형·이웃·가리지 않은 칸에서 무언가를 배운 것입니다.
학습 마디가 48개 안팎이라 100–150 step 뒤로는 학습 ELBO가 0으로 가고 보류 ELBO가 오르는 과적합이 정상적으로 나타납니다.

**1) 가운데: 구조 실현(감시).** 끝까지 생성한 곡에서
- `유형 소속(거부 전)`: 각 마디가 조건으로 준 유형의 대표형에 가장 가까운 비율(편집거리). 거부로 고치기 전의 값이라 모형 자체의 능력입니다.
- `표면 전이가 G에 있음`: 음표로 다시 읽은 이웃 마디의 유형 전이가 원곡 그래프에 있는 비율. 10월 MaskGIT은 약 0.61이었습니다.

`diffusion_graph`가 `diffusion`보다 높으면 그래프 손실이 "국소 생성이 상정한 구조를 실현하게" 만든 것입니다.

**1) 오른쪽: 원곡 마디 복사율.** 높을수록 새 선율이 아니라 원곡 마디를 골라 붙인 것입니다. 선율이 새로워야 한다는 기준에서 가장 먼저 볼 값입니다.

**3) 생성 시 기록한 지표.**
- 구조: `first_pass_membership`(거부 전 유형 소속), `membership`(거부 후), `surface_edge_validity`, `surface_graph_tv`(표면 그래프와 조건 그래프의 총변동 거리, 0이면 같음), `returns_exact`(구성상 1).
- 새로움: `walk_positions_differing`(유형열이 원곡과 다른 자리 비율), `original_phrase_rate`, `unseen_note_transition_rate`, `note_4gram_novelty`.
- 양식(참고): `interval_jsd`, `duration_jsd`.

**5)–7) 평가 (고정된 설계).**
- 기하: `phrase_graph_tv`(곡마다 표면 유형 그래프와 원곡 G의 TV), `pooled_phrase_graph_tv`(60곡 합산), `ph_bottleneck`·`ph_wasserstein`(음표 그래프의 1차원 barcode를 원곡과 비교, 정재훈 교수팀이 쓰는 기하), `ph_cycles`(원곡은 9개).
- 통계: 음정·음가·음높이·음 3/4-gram 분포의 JSD(작을수록 원곡과 비슷), `oa_*`(마디 사이 편집거리 분포의 겹침 넓이, 클수록 비슷).
- 음악: `pitch_outside_scale`·`beat_fill_error`·`offgrid_onset`은 격자 모형에서 구성상 0입니다(다른 생성기와 비교할 때 의미가 있음). `chapter_cadence_ok`(장 끝 종지음이 원곡 종지음 집합 안), `final_pitch_ok`.
- 새로움: `exact_copy_rate`, `authenticity`, `density`, `coverage`, `nn_distance_mean`(원곡 마디 안 최근접 거리 평균은 원곡 기준값의 `nn1_mean`), `longest_shared_run`(원곡과 연속으로 같은 음표 수).
  `exact_copy_rate`는 원곡 기준값의 `repeat_rate_free`(원곡 자체가 곡 안에서 반복하는 마디 비율, 약 0.43)와 비교해 읽습니다.
  coverage는 곡 단위 bootstrap이 아래로 치우쳐서 구간을 내지 않습니다.
- 복제 검정: `z`(전부)와 `z_novel`(학습 마디와 똑같은 마디를 뺀 나머지). 판정 기준 4의 |Z| < 2는 `z_novel`로 봅니다.

결과 폴더(`DiffDodree_chart/artifacts/chart_diffusion_runs/<RUN_TAG>/`)를 알려 주시면 Mac에서 읽어 해석하겠습니다.